In [1]:
import os
import cv2
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input, Dropout
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

def load_data():
  from google.colab import drive
  drive.mount('/content/drive')
  dataset_path = "/content/drive/MyDrive/ALL2upload"

  images = []
  filenames = []
  labels = []

  for label in ["Malignant", "Benign"]:
    folder_path = os.path.join(dataset_path, label)

    for filename in os.listdir(folder_path):
      image_path = os.path.join(folder_path, filename)
      image = cv2.imread(image_path) #converts to array

      if image is None:
        continue

      images.append(image)
      filenames.append(filename)

      if label == 'Malignant':
        labels.append(1)
      else:
        labels.append(0)

  return images,labels,filenames

def preprocess(images_list):
  #hsv range
  converted_images=[]
  lower = np.array([118, 35, 204])
  upper = np.array([143, 230, 255])

  #preprocessing steps
  for im in images_list:
    hsv = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv,lower,upper)
    kernel = np.ones((3,3),dtype=np.uint8)
    mask = cv2.morphologyEx(mask,cv2.MORPH_OPEN,kernel)
    mask = cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel)
    result = cv2.bitwise_and(im,im,mask=mask)
    white = np.full(im.shape, 255, dtype=np.uint8)
    inverted_mask = cv2.bitwise_not(mask)
    cutout = cv2.bitwise_and(white,white, mask=inverted_mask)
    final_output = cv2.bitwise_or(cutout,result)
    converted_images.append(final_output)

  return converted_images

def traintest_classweight_normalize(images_list,labels_list,seed):
  #Train test split
  x = np.array(images_list)
  y = np.array(labels_list)
  x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=seed)

  #Class weights
  weights = compute_class_weight(
      class_weight = 'balanced',
      classes= np.unique(y_train),
      y=y_train
  )

  class_weights={
      0: weights[0],
      1: weights[1]
  }

  #Normalize
  x_test = x_test/255
  x_train = x_train/255

  return x_train, y_train, x_test, y_test, class_weights

def train_model(x_train, y_train, class_weights):
  #CNN
  model = Sequential([
      Input(shape = (224,224,3)),

      Conv2D(32,(3,3), activation='relu', name='conv1'),
      MaxPooling2D((2,2)),

      Conv2D(64,(3,3), activation='relu', name='conv2'),
      MaxPooling2D((2,2)),

      Conv2D(128,(3,3), activation='relu', name='conv3'),
      MaxPooling2D((2,2)),

      Flatten(),
      Dropout(0.25),
      Dense(128, activation='relu'),
      Dense(1, activation='sigmoid')
  ])

  #Configure training
  optimizer = AdamW(learning_rate=1e-4)

  model.compile(
      optimizer=optimizer,
      loss='binary_crossentropy',
      metrics = ['accuracy']
  )

  #Train
  model.fit(x_train, y_train, epochs=15, class_weight=class_weights,verbose=0)

  return model

def evaluate_model(model, x_test, y_test):
  #Test & print accuracy
  loss, accuracy = model.evaluate(x_test,y_test)
  print(f'Accuracy: {accuracy}')
  y_pred = model.predict(x_test, verbose=0)
  y_pred_binary = y_pred.round()

  #Confusion matrix
  cm = confusion_matrix(y_test,y_pred_binary)
  tn, fp, fn, tp = cm.ravel()
  #print(f'TN: {tn} FP: {fp} FN: {fn} TP: {tp}') in case I want to change display type
  print("TN:", tn)
  print("FP:", fp)
  print("FN:", fn)
  print("TP:", tp)

  #Classification report
  report = classification_report(y_test, y_pred_binary, output_dict=True)
  print(f"Overall Accuracy: {report['accuracy']:.3f}")
  print()

  print('Benign Stats')
  print(f'Precision: {report["0"]["precision"]:.3f}')
  print(f'Recall: {report["0"]["recall"]:.3f}')
  print(f'F1: {report["0"]["f1-score"]:.3f}')
  print()

  print('Malignant Stats')
  print(f'Precision: {report["1"]["precision"]:.3f}')
  print(f'Recall: {report["1"]["recall"]:.3f}')
  print(f'F1: {report["1"]["f1-score"]:.3f}')
  print()

def grad_cam(model,x_test, random_grad_idx):
  image = x_test[random_grad_idx]
  image = np.expand_dims(image, axis=0)
  last_conv_layer = model.get_layer('conv3')

  # Get the final Dense layer weights
  final_layer = model.layers[-1]
  weights, bias = final_layer.get_weights()

  grad_model = tf.keras.models.Model(
    inputs = model.inputs,
    outputs=[
        last_conv_layer.output,
        model.layers[-2].output
    ])

  with tf.GradientTape() as tape:
    last_conv_output, dense_features = grad_model([image])
    logit = tf.matmul(dense_features, weights) + bias

  grads = tape.gradient(logit, last_conv_output)
  importance_weights = tf.reduce_mean(grads, axis=(0,1,2))
  last_conv_output = last_conv_output[0]
  heatmap = last_conv_output * importance_weights
  heatmap = tf.reduce_sum(heatmap, axis=-1)
  heatmap = tf.maximum(heatmap, 0)

  max_value = tf.reduce_max(heatmap) # prevents division by zero
  if max_value > 0:
    heatmap = heatmap / max_value

  heatmap = heatmap.numpy()
  heatmap = cv2.resize(heatmap, (224, 224))
  heatmap = np.uint8(255 * heatmap)

  return heatmap

def display_visual(heatmap, random_grad_idx, x_test, y_test, model):
  print('Displaying one sample image')

  test_image = x_test[random_grad_idx]
  original_image = (test_image * 255).astype(np.uint8)

  color_heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
  superimposed = cv2.addWeighted(original_image, 0.6, color_heatmap, 0.4, 0)

  fig, axes = plt.subplots(1, 4, figsize=(16, 4))
  axes[0].imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
  axes[0].set_title("Original")
  axes[1].imshow(heatmap, cmap='gray')
  axes[1].set_title("Gray Map")
  axes[2].imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
  axes[2].set_title("Grad-CAM overlay")
  axes[3].imshow(heatmap, cmap="jet")
  axes[3].set_title("Raw heatmap")

  plt.tight_layout()
  plt.show()

  #To see if it was correct or not
  image = np.expand_dims(test_image, axis=0)
  prediction = model.predict(image, verbose=0)

  print("True label:", y_test[random_grad_idx])
  print("Raw prediction:", prediction[0][0])
  rounded = prediction[0][0]
  rounded = rounded.round()
  print(f'Rounded result: {rounded}')
  print(f'Index: {random_grad_idx}')
  print()


In [ ]:
#Main Program

images_list, labels_list, filenames_list = load_data()
seeds = [42,43,44,45,46,47,48,49,50,51]

#PIPELINE 1
for seed in seeds:
  np.random.seed(seed)
  tf.random.set_seed(seed)

  x_train, y_train, x_test, y_test, class_weights = traintest_classweight_normalize(images_list, labels_list,seed)
  model = train_model(x_train, y_train, class_weights)
  print(f'Pipeline 1 results with seed = {seed}')
  evaluate_model(model, x_test, y_test)

  random_grad_idx = random.randint(0, len(x_test) - 1)

  heatmap = grad_cam(model, x_test, random_grad_idx)
  display_visual(heatmap, random_grad_idx, x_test, y_test, model)

#PIPELINE 2:
converted_images = preprocess(images_list)

for seed in seeds:
  np.random.seed(seed)
  tf.random.set_seed(seed)

  x_train, y_train, x_test, y_test, class_weights = traintest_classweight_normalize(converted_images, labels_list, seed)
  model = train_model(x_train, y_train, class_weights)
  print(f'Pipeline 2 results with seed = {seed}')
  evaluate_model(model, x_test, y_test)

  random_grad_idx = random.randint(0, len(x_test) - 1)

  heatmap = grad_cam(model, x_test, random_grad_idx)
  display_visual(heatmap, random_grad_idx, x_test, y_test, model)


In [ ]:
#DEMO PREDICTION
answer_for_demo = input('Would you like to continue (yes/no)')
if answer_for_demo == 'no':
  sys.exit()
elif answer_for_demo != 'yes':
  print('invalid answer')
  sys.exit()

random_grad_idx = random.randint(0, len(x_test) - 1)
display_visual(heatmap, random_grad_idx, x_test, y_test, model)

answer = input('Would you like to try again with another image? (yes/no)')
while answer == 'yes':
  random_grad_idx = random.randint(0, len(x_test) - 1)
  heatmap = grad_cam(model, x_test, random_grad_idx)
  display_visual(heatmap, random_grad_idx, x_test, y_test, model)
  answer = input('Would you like to try again with another image? (yes/no)')